# 20 — Analysis (Stage 3)

Pure pandas over `results_v1.jsonl`; **never calls an API**. Narrative order mirrors spec §6: noise floor & placebo gate first, then per-axis results, localization (flip rates), verbosity, self-preference, the validity anchor, and issue matching. Figures/tables are written to `artifacts/figures/`.

Set `RUN_NAME` below — use `results_v1` for the real run (or a mock run to dry-run the whole notebook before any spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))  # run from notebooks/
from prjudge.config import load_config
config = load_config()
from prjudge import analysis as A
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 160)
RUN_NAME = 'results_v1'   # or a mock run name to validate the notebook first
FIG = A.figures_dir(config)
df = A.load_results(config, RUN_NAME)
print('scored rows:', len(df), '| unparsed:', A.load_results.n_unparsed)
print('judges:', sorted(df['judge'].unique()), '| variants:', df['variant'].nunique())

## 1. Noise floor & placebo gate
Every reported flip rate and Δ is compared against trial-to-trial noise. The **placebo axis must show ≈0 effect** — if it doesn't, the noise model is wrong and results are not reported until resolved.

In [ ]:
nf = A.noise_floor(df)
display(nf)
gate = A.placebo_gate(df)
print('PLACEBO GATE:', 'PASS' if gate['passes'] else 'FAIL')
print(gate)

## 2. Primary — Δ(aggregate) per axis
Δ = variant − same-PR baseline, paired within (PR, judge). Axes ranked by median |Δ| per judge, with Wilcoxon and a PR-random-intercept mixed model.

In [ ]:
summary = A.delta_summary(df)
display(summary)
display(A.wilcoxon_by_axis(df))
mm = A.mixedlm_by_axis(df)
display(mm if not mm.empty else 'statsmodels unavailable')

In [ ]:
# Figure: median |Δ| by axis, per judge
piv = summary.pivot(index='axis', columns='judge', values='median_abs_delta')
ax = piv.plot(kind='barh', figsize=(8, 4))
ax.set_xlabel('median |Δ aggregate| (0–10 scale)'); ax.set_title('Perturbation sensitivity by axis')
plt.tight_layout(); plt.savefig(FIG / 'axis_sensitivity.png', dpi=150); plt.show()

## 3. Bias localization — per-item flip rates
Which judgments absorb each perturbation. Padding flipping item 8 (understandability) is semi-defensible; flipping item 1 or 3 — code the padding never touched — is the damning finding. The most quotable output.

In [ ]:
fr = A.flip_rates(df)
heat = fr.pivot_table(index='item_text', columns='axis', values='flip_rate', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(heat.values, aspect='auto', cmap='magma')
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=30, ha='right')
ax.set_yticks(range(len(heat.index))); ax.set_yticklabels(heat.index, fontsize=8)
fig.colorbar(im, label='flip rate'); ax.set_title('Per-item flip rate by axis')
plt.tight_layout(); plt.savefig(FIG / 'flip_rate_heatmap.png', dpi=150); plt.show()
display(heat.round(3))

## 4. Verbosity dose–response
Monotonic score-vs-length trend across terse < baseline < 2× < 4×, with the sensitivity check excluding the 4 short-description PRs. The `verb_terse` cells of the 4 terse-waived PRs (spec §4.2) are excluded by default (`exclude_terse_waived=True`).

In [ ]:
print('all PRs:'); display(A.verbosity_trend(df))
print('excluding short-desc PRs:'); display(A.verbosity_trend(df, exclude_short_desc=True))

In [ ]:
vt = A.verbosity_trend(df)
order = A.VERBOSITY_ORDER
fig, ax = plt.subplots(figsize=(7, 4))
for _, r in vt.iterrows():
    ax.plot(order, [r[f'mean_{v}'] for v in order], marker='o', label=r['judge'])
ax.set_ylabel('mean aggregate (0–10)'); ax.set_xlabel('verbosity dose')
ax.set_title('Verbosity dose–response'); ax.legend()
plt.tight_layout(); plt.savefig(FIG / 'verbosity_trend.png', dpi=150); plt.show()

## 5. Self-preference 2×2
Claimed-family × judge-family. The interaction (own-family minus other-family Δ) is the self-preference estimate.

In [ ]:
display(A.self_preference(df))

## 6. Validity anchor — item 9 vs human `has_requested_changes`
Robustness is necessary, not sufficient: a judge with near-zero sensitivity **and** near-zero validity is vacuously stable. Direct binary agreement plus Spearman of the baseline aggregate with `requested_change_count`.

In [ ]:
display(A.item9_validity(df))

## 7. Issue matching — detection overlap & shift
Judge-reported issues matched to human-flagged locations (same file, line in the comment's `diff_hunk` range). Baseline = validity evidence; under perturbation, a detection drop is the mechanism story.

In [ ]:
im = A.issue_matching(config, df)
display(im.pivot(index='variant', columns='judge', values='detection').round(3))

## 8. 2D judge characterization
Each judge on validity × worst-case bias magnitude. The sensitive+valid quadrant is the most dangerous deployment profile: right on average, swayed by surface features.

In [ ]:
val = A.item9_validity(df).set_index('judge')['item9_vs_human_agreement']
bias = A.delta_summary(df).groupby('judge')['median_abs_delta'].max()
fig, ax = plt.subplots(figsize=(6, 5))
for j in val.index:
    ax.scatter(val[j], bias.get(j, np.nan), s=80)
    ax.annotate(j, (val[j], bias.get(j, np.nan)), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('validity (item-9 agreement)'); ax.set_ylabel('worst-case median |Δ|')
ax.set_title('Judge characterization: validity × bias')
plt.tight_layout(); plt.savefig(FIG / 'judge_characterization.png', dpi=150); plt.show()